# 01 — Data Prep

Download, verify, clean, and join everything downstream notebooks depend on.

**Outputs**
- `results/data_verification.csv` — verification gate table (graded requirement)
- `results/data_quality_summary.csv`, `results/data_missingness.csv`
- `data/processed/{specs,sales,mobiles,reviews_global,reviews_background}_clean.csv`

**Known dataset facts** (verified 2026-10-19, see `build_plan.md` section 3):
- `reviews.csv` (masaladata, 7 files) ends **May 2017** — it cannot contain foldable
  reviews (first foldable shipped 2019). It is a background corpus only.
- `reviews_global.csv` is the **primary foldable review corpus**.
- `specs.csv` price column has no declared currency: treated as **USD (assumption,
  documented below and cross-checked against `mobiles_2025` US launch prices).**



## 0. Setup

Paths resolve whether the notebook is run from `notebooks/` or the project root.
The FX table is a **fixed, documented assumption** (build plan section 3): update the
rates before the final submission run and re-run top-to-bottom.



In [ ]:
from __future__ import annotations

import re
from pathlib import Path

import numpy as np
import pandas as pd

# Paths: notebook lives in project/notebooks/; data and results sit at project/ level.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW = PROJECT_ROOT / "data" / "raw"
PROCESSED = PROJECT_ROOT / "data" / "processed"
ENRICHMENT = PROJECT_ROOT / "data" / "enrichment"
RESULTS = PROJECT_ROOT / "results"
for d in (PROCESSED, RESULTS):
    d.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 160)

# Fixed currency conversion table (documented assumption).
# Approximate rates as of 2026-10; update before the final submission run.
FX_ASOF = "2026-10-19"
EUR_PER_UNIT = {
    "USD": 0.92,   # assumed currency of the specs.csv price column - cross-checked below
    "PKR": 0.0032,
    "INR": 0.0104,
    "CNY": 0.128,
    "AED": 0.250,  # Dubai
}

# is_foldable rule: word-boundary match on fold / flip / razr, case-insensitive.
# "duo" is deliberately EXCLUDED: Microsoft Surface Duo is a dual-screen device,
# not a folding-panel one (build plan). Hyphens and slashes are word boundaries,
# so URL slugs like "samsung-galaxy-z-fold" match correctly.
FOLDABLE_RE = re.compile(r"\b(fold|flip|razr)\b", flags=re.IGNORECASE)



## 1. Load raw datasets

Files are resolved by name pattern so both the plan filenames (`specs.csv`) and the
original Kaggle filenames (`smartphones.csv`, `Sales.csv`,
`phone_user_review_file_1..7.csv`) work. Missing files raise with a download hint.



In [ ]:
def find_raw(*patterns):
    """First CSV in data/raw/ whose name contains any pattern (case-insensitive)."""
    for pat in patterns:
        hits = sorted(p for p in RAW.glob("*.csv") if pat.lower() in p.name.lower())
        if hits:
            return hits[0]
    return None


def load_csv(name, *patterns, **kwargs):
    """Load a raw dataset by filename pattern; raise with a download hint if absent."""
    path = find_raw(*patterns)
    if path is None:
        raise FileNotFoundError(
            f"{name}: no CSV matching {patterns} in {RAW}.\n"
            "Download it per README.md 'Dataset sources' and place it in data/raw/."
        )
    df = pd.read_csv(path, **kwargs)
    print(f"[load] {name:<16} <- {path.name:<38} {df.shape[0]:>8,} rows x {df.shape[1]:>2} cols")
    return df


def col(df, *candidates):
    """Case/spacing/punctuation-insensitive column lookup. Actual name or None."""
    norm = {re.sub(r"[^a-z0-9]", "", str(c).lower()): c for c in df.columns}
    for cand in candidates:
        key = re.sub(r"[^a-z0-9]", "", cand.lower())
        if key in norm:
            return norm[key]
    return None



In [ ]:
specs = load_csv("specs.csv", "specifications", "smartphones", "specs")
sales = load_csv("sales_india.csv", "sales_india", "Sales")
mobiles = load_csv("mobiles_2025.csv", "mobiles_2025", "mobiles")
reviews_global = load_csv("reviews_global.csv", "reviews_global", low_memory=False)

# reviews.csv (masaladata) ships as 7 files; concatenate all that are present.
rev_files = sorted(RAW.glob("phone_user_review_file_*.csv")) or \
    sorted(p for p in RAW.glob("*.csv") if "cell_phone_review" in p.name.lower())
if rev_files:
    reviews = pd.concat(
        (pd.read_csv(f, low_memory=False) for f in rev_files), ignore_index=True
    )
    print(f"[load] reviews.csv      <- {len(rev_files)} concatenated files       "
          f"{len(reviews):>8,} rows x {reviews.shape[1]:>2} cols")
else:
    reviews = None
    print("[load] reviews.csv      not found (background corpus only) - skipping")



## 2. Verification gate (graded requirement)

Rules (build plan section 3, as amended):
- **records > 1,000** for every dataset,
- **features >= 25** for `specs.csv` only (review/sales datasets naturally have
  5–15 columns; applying 25 there would trigger false failures).

Hard-fail applies only to datasets that have a defined backup (specs, reviews,
reviews_global). For the others a failure is a loud warning — the group must
decide whether to substitute data before submission.



In [ ]:
GATE_MIN_RECORDS = 1_000
GATE_MIN_FEATURES_SPECS = 25

gate_rows = []

def gate(name, df, feature_rule=False, hard=True):
    n_rows, n_cols = df.shape
    rec_ok = n_rows > GATE_MIN_RECORDS
    feat_ok = (not feature_rule) or n_cols >= GATE_MIN_FEATURES_SPECS
    gate_rows.append({
        "dataset": name, "records": n_rows, "features": n_cols,
        "records_ok": rec_ok, "features_ok": feat_ok,
        "passes": rec_ok and feat_ok, "hard": hard,
    })

gate("specs.csv", specs, feature_rule=True)
gate("reviews.csv", reviews, hard=False) if reviews is not None else None
gate("reviews_global.csv", reviews_global)
gate("mobiles_2025.csv", mobiles, hard=False)
gate("sales_india.csv", sales, hard=False)

verification = pd.DataFrame(gate_rows)
display(verification)
verification.to_csv(RESULTS / "data_verification.csv", index=False)

failed_hard = verification.loc[~verification["passes"] & verification["hard"], "dataset"].tolist()
failed_soft = verification.loc[~verification["passes"] & ~verification["hard"], "dataset"].tolist()
assert not failed_hard, (
    f"Verification gate FAILED for {failed_hard}. Swap in the backup dataset "
    "per build_plan.md section 3, then re-run."
)
if failed_soft:
    print(f"Verification gate WARNING (no backup defined): {failed_soft} "
          "below threshold - group must decide before submission.")
print("Verification gate: done. Table saved to results/data_verification.csv")



## 3. Cleaning

Per dataset: soft numeric coercion (specs ship strings like `"8 GB"`), missing-value
drops on key columns, duplicate model removal, IQR outlier removal on price.



In [ ]:
def to_numeric_soft(df):
    """Convert string columns that are >= 80% parseable to numbers ('8 GB' -> 8)."""
    for c in list(df.columns):
        if df[c].dtype == object:
            conv = pd.to_numeric(
                df[c].astype(str).str.extract(r"(\d+\.?\d*)", expand=False),
                errors="coerce",
            )
            if conv.notna().mean() >= 0.8:
                df[c] = conv
    return df


def drop_iqr_outliers(df, c):
    """Remove rows where column c is outside [Q1 - 1.5*IQR, Q3 + 1.5*IQR]."""
    s = df[c].dropna()
    if s.empty:
        return 0, df
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    lo, hi = q1 - 1.5 * (q3 - q1), q3 + 1.5 * (q3 - q1)
    keep = df[c].isna() | df[c].between(lo, hi)
    return int((~keep).sum()), df[keep]


quality_rows = []

def note(dataset, rows_in, rows_out, extra=""):
    quality_rows.append({"dataset": dataset, "rows_in": rows_in,
                         "rows_out": rows_out, "note": extra})



In [ ]:
# ---- specs.csv ----
n_in = len(specs)
specs = to_numeric_soft(specs)
model_c, brand_c, price_c = col(specs, "model"), col(specs, "brand"), col(specs, "price")
drop_on = [c for c in (model_c, price_c) if c]
if drop_on:
    specs = specs.dropna(subset=drop_on)
if model_c and brand_c:
    specs = specs.drop_duplicates(subset=[brand_c, model_c])
if price_c:
    n_out, specs = drop_iqr_outliers(specs, price_c)
    print(f"[clean] specs: dropped {n_out} price outliers (IQR)")
note("specs.csv", n_in, len(specs))
print(f"[clean] specs.csv: {n_in:,} -> {len(specs):,} rows")

# ---- mobiles_2025.csv ----
n_in = len(mobiles)
mobiles = to_numeric_soft(mobiles)
m_model_c, m_brand_c = col(mobiles, "model"), col(mobiles, "company", "brand")
if m_model_c:
    mobiles = mobiles.dropna(subset=[m_model_c])
if m_model_c and m_brand_c:
    mobiles = mobiles.drop_duplicates(subset=[m_brand_c, m_model_c])
note("mobiles_2025.csv", n_in, len(mobiles))
print(f"[clean] mobiles_2025.csv: {n_in:,} -> {len(mobiles):,} rows")

# ---- sales_india.csv ----
n_in = len(sales)
sales = to_numeric_soft(sales)
s_brand_c = col(sales, "brands", "brand")
s_price_c = col(sales, "selling price", "price")
if s_price_c:
    sales = sales.dropna(subset=[s_price_c])
s_model_c = col(sales, "model", "mobile")
if s_brand_c and s_model_c:
    sales = sales.drop_duplicates(subset=[s_brand_c, s_model_c])
elif s_brand_c:
    sales = sales.drop_duplicates(subset=[s_brand_c])
note("sales_india.csv", n_in, len(sales))
print(f"[clean] sales_india.csv: {n_in:,} -> {len(sales):,} rows")



In [ ]:
# ---- reviews_global.csv (PRIMARY foldable corpus) ----
n_in = len(reviews_global)
rg_model_c = col(reviews_global, "model", "phone", "product")
rg_text_c = col(reviews_global, "review", "review text", "comment", "text")
if rg_text_c:
    reviews_global = reviews_global.dropna(subset=[rg_text_c])
if rg_model_c and rg_text_c:
    reviews_global = reviews_global.drop_duplicates(subset=[rg_model_c, rg_text_c])
note("reviews_global.csv", n_in, len(reviews_global))
print(f"[clean] reviews_global.csv: {n_in:,} -> {len(reviews_global):,} rows")

# ---- reviews.csv (background corpus; reduce to essential columns) ----
if reviews is not None:
    n_in = len(reviews)
    keep = [c for c in (
        col(reviews, "phone url", "phone", "slug"),   # model identifier (URL slug)
        col(reviews, "date"),
        col(reviews, "score", "rating"),
        col(reviews, "extract", "review", "text"),   # review body
        col(reviews, "country"),
        col(reviews, "language", "lang"),
    ) if c]
    reviews = reviews[keep].dropna(
        subset=[c for c in (col(reviews, "extract", "review", "text"),) if c]
    ).drop_duplicates()
    note("reviews.csv", n_in, len(reviews), extra="reduced columns; ends 2017, background only")
    print(f"[clean] reviews.csv: {n_in:,} -> {len(reviews):,} rows (background corpus)")



## 4. Price standardization to EUR

Fixed conversion table (top cell), rate date `FX_ASOF` documented. Assumption under
review: **specs.csv prices are treated as USD**. Sanity check: median `specs` EUR
price vs median `mobiles_2025` US launch price in EUR should be the same order of
magnitude; if not, revisit the assumption before notebooks 02/03.



In [ ]:
def to_eur(series, currency):
    return (series * EUR_PER_UNIT[currency]).round(2)

# specs: assumed USD (documented assumption, cross-checked below).
specs_price_c = col(specs, "price")
if specs_price_c:
    specs["price_eur"] = to_eur(specs[specs_price_c], "USD")

# mobiles: one EUR column per per-country launch price column.
CURRENCY_TOKENS = {"pakistan": "PKR", "india": "INR", "china": "CNY",
                   "usa": "USD", "dubai": "AED", "uae": "AED"}
for c in list(mobiles.columns):
    lc = c.lower()
    if "price" in lc and mobiles[c].dtype != object:
        for token, cur in CURRENCY_TOKENS.items():
            if token in lc:
                mobiles[f"{c}_eur"] = to_eur(mobiles[c], cur)
                break

# sales: INR.
for c in list(sales.columns):
    if "price" in c.lower() and sales[c].dtype != object:
        sales[f"{c}_eur"] = to_eur(sales[c], "INR")

# Currency sanity check for the specs-as-USD assumption.
us_eur_col = next((c for c in mobiles.columns if c.endswith("_eur") and "usa" in c.lower()), None)
if specs_price_c and us_eur_col:
    print(f"[fx] specs price_eur median: {specs['price_eur'].median():,.0f} EUR")
    print(f"[fx] mobiles US  _eur median: {mobiles[us_eur_col].median():,.0f} EUR")
    print("[fx] Same order of magnitude -> USD assumption holds; "
          "otherwise revisit before notebooks 02/03.")
print(f"[fx] EUR conversion applied (rates as of {FX_ASOF}; update before final run).")



## 5. `is_foldable` flag — the project-wide join key

Word-boundary match on **fold / flip / razr** (see top cell). Sample matches are
printed for manual inspection — the NLP notebooks depend on this flag being right.



In [ ]:
def add_foldable_flag(df, *name_cols):
    c = None
    for n in name_cols:
        c = col(df, n)
        if c:
            break
    if c is None:
        df["is_foldable"] = False
        print(f"[flag] WARNING: no model column among {name_cols}; flag set to False")
        return df
    df["is_foldable"] = df[c].astype(str).str.contains(FOLDABLE_RE)
    print(f"[flag] {df.shape[0]:>8,} rows | foldables: {df['is_foldable'].sum():>6,} "
          f"({100 * df['is_foldable'].mean():.2f}%)  [matched on '{c}']")
    return df

specs = add_foldable_flag(specs, "model")
mobiles = add_foldable_flag(mobiles, "model")
reviews_global = add_foldable_flag(reviews_global, "model", "phone", "product")
if reviews is not None:
    reviews = add_foldable_flag(reviews, "phone url", "phone", "slug")

print("\n[flag] sample foldable matches in specs.csv:")
print(specs.loc[specs["is_foldable"], col(specs, "model")].head(10).tolist())
print("\n[flag] sample foldable matches in reviews_global.csv:")
rg_model_c = col(reviews_global, "model", "phone", "product")
print(reviews_global.loc[reviews_global["is_foldable"], rg_model_c].head(10).tolist())
print("\nHONESTY NOTE: reviews.csv ends May 2017 -> its foldable count above is "
      "expected to be 0 by construction. reviews_global.csv is the foldable corpus.")



## 6. Enrichment joins

- `enrichment_market.csv` (compiled from axis-intelligence.com, cited): foldable
  market stats joined onto `specs` by launch year.
- `enrichment_wb.csv` (World Bank DataBank): mobile penetration joined onto
  `reviews_global` by country.

Both files are compiled by the group; if absent the notebook warns and continues so
development is not blocked — but the final run requires them (acceptance criterion 6).



In [ ]:
mkt_path = ENRICHMENT / "enrichment_market.csv"
if mkt_path.exists():
    mkt = pd.read_csv(mkt_path)
    mkt_year_c = col(mkt, "year")
    specs_year_c = col(specs, "year")
    if mkt_year_c and specs_year_c:
        specs = specs.merge(mkt, left_on=specs_year_c, right_on=mkt_year_c,
                            how="left", suffixes=("", "_mkt"))
        print(f"[enrich] joined market stats ({mkt.shape[1]} cols) onto specs by launch year")
    else:
        print("[enrich] WARNING: enrichment_market.csv lacks a 'year' column; join skipped")
else:
    print("[enrich] WARNING: enrichment_market.csv not found - compile per build plan section 3")

wb_path = ENRICHMENT / "enrichment_wb.csv"
if wb_path.exists():
    wb = pd.read_csv(wb_path)
    wb_country_c = col(wb, "country")
    rg_country_c = col(reviews_global, "country")
    if wb_country_c and rg_country_c and wb.shape[1] >= 2:
        value_c = next(c for c in wb.columns if c != wb_country_c)
        mapping = dict(zip(
            wb[wb_country_c].astype(str).str.strip().str.lower(),
            wb[value_c],
        ))
        reviews_global["country_penetration"] = (
            reviews_global[rg_country_c].astype(str).str.strip().str.lower().map(mapping)
        )
        matched = reviews_global["country_penetration"].notna().mean()
        print(f"[enrich] World Bank penetration joined onto reviews_global "
              f"({100 * matched:.0f}% of rows matched)")
    else:
        print("[enrich] WARNING: enrichment_wb.csv lacks 'country' column; join skipped")
else:
    print("[enrich] WARNING: enrichment_wb.csv not found - compile per build plan section 3")



## 7. Save outputs

Processed tables go to `data/processed/`; the quality summary and missingness table
go to `results/`. Downstream notebooks load ONLY from `data/processed/`.



In [ ]:
outputs = {
    "specs_clean.csv": specs,
    "sales_clean.csv": sales,
    "mobiles_clean.csv": mobiles,
    "reviews_global_clean.csv": reviews_global,
}
if reviews is not None:
    outputs["reviews_background_clean.csv"] = reviews

for fname, df in outputs.items():
    df.to_csv(PROCESSED / fname, index=False)
    print(f"[save] {fname:<30} {len(df):>8,} rows x {df.shape[1]:>2} cols")

quality = pd.DataFrame(quality_rows)
quality.to_csv(RESULTS / "data_quality_summary.csv", index=False)

missingness = []
for name, df in outputs.items():
    for c in df.columns:
        missingness.append({"dataset": name, "column": c,
                            "missing_pct": round(100 * df[c].isna().mean(), 2)})
pd.DataFrame(missingness).to_csv(RESULTS / "data_missingness.csv", index=False)
print(f"[save] results/data_quality_summary.csv ({len(quality)} rows)")
print(f"[save] results/data_missingness.csv    ({len(missingness)} rows)")



## Next steps

1. **Currency assumption**: verify the specs-as-USD cross-check output above before
   notebooks 02/03 are trusted.
2. **`is_foldable` inspection**: eyeball the sample matches; extend the regex only
   with a documented justification.
3. **Enrichment**: compile `enrichment_market.csv` and `enrichment_wb.csv`, then
   re-run this notebook so the final `results/` are complete.
4. Then build notebook 02 (price regression) on `specs_clean.csv`.
